## Set up

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd
import os
DRIVE_DIR = '/content/drive/MyDrive/Bad Mood Prediction/data'
df_daily = pd.read_parquet(f'{DRIVE_DIR}/parquet/rais_anonymized__csv_rais_anonymized__daily_fitbit_sema_df_unprocessed.parquet')
df_hourly = pd.read_parquet(f'{DRIVE_DIR}/parquet/rais_anonymized__csv_rais_anonymized__hourly_fitbit_sema_df_unprocessed.parquet')

Mounted at /content/drive


In [ ]:
import glob
import os
import pandas as pd

PARQUET_DIR = f'{DRIVE_DIR}/parquet'
os.makedirs(PARQUET_DIR, exist_ok=True)

for csv in glob.glob('/content/lifesnaps/**/*.csv', recursive=True):
    name = os.path.splitext(os.path.basename(csv))[0]
    out = f'{PARQUET_DIR}/{name}.parquet'

    if not os.path.exists(out):
        print(f'Converting {name}...')

        # Read the CSV with consistent type inference
        df = pd.read_csv(csv, low_memory=False)

        # Make mixed-type columns consistent for Parquet
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].map(
                lambda x: str(x) if pd.notna(x) else None
            )

        df.to_parquet(out, index=False)

        print(f'Saved {out}')

print('Conversion complete!')


Conversion complete!


In [ ]:
PARQUET_DIR = f'{DRIVE_DIR}/parquet'

daily_path = f'{PARQUET_DIR}/rais_anonymized__csv_rais_anonymized__daily_fitbit_sema_df_unprocessed.parquet'
hourly_path = f'{PARQUET_DIR}/rais_anonymized__csv_rais_anonymized__hourly_fitbit_sema_df_unprocessed.parquet'

daily = pd.read_parquet(daily_path)
hourly = pd.read_parquet(hourly_path)

print("Daily shape:", daily.shape)
print("Hourly shape:", hourly.shape)

display(daily.head())
display(hourly.head())


Daily shape: (7410, 63)
Hourly shape: (159508, 39)


,Unnamed: 0,id,date,nightly_temperature,nremhr,rmssd,spo2,full_sleep_breathing_rate,stress_score,sleep_points_percentage,...,TENSE/ANXIOUS,TIRED,ENTERTAINMENT,GYM,HOME,HOME_OFFICE,OTHER,OUTDOORS,TRANSIT,WORK/SCHOOL
0,0,621e2e8e67b776a24055b564,2021-05-24,34.137687,57.432,89.603,NaN,14.8,78.0,0.833333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,621e2e8e67b776a24055b564,2021-05-25,33.794544,57.681,94.303,NaN,15.8,80.0,0.833333,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
2,2,621e2e8e67b776a24055b564,2021-05-26,34.611011,57.481,119.212,NaN,14.6,84.0,0.966667,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,3,621e2e8e67b776a24055b564,2021-05-27,34.408304,57.493,111.709,NaN,14.8,82.0,0.933333,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
4,4,621e2e8e67b776a24055b564,2021-05-28,34.178922,56.750,103.034,NaN,15.2,81.0,0.866667,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


,Unnamed: 0,id,date,hour,temperature,badgeType,calories,distance,activityType,bpm,...,TENSE/ANXIOUS,TIRED,ENTERTAINMENT,GYM,HOME,HOME_OFFICE,OTHER,OUTDOORS,TRANSIT,WORK/SCHOOL
0,0,621e2e8e67b776a24055b564,2021-05-24,0.0,-1.466659,<NA>,89.04,98.3,<NA>,66.874763,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,621e2e8e67b776a24055b564,2021-05-24,1.0,0.210289,<NA>,65.62,0.0,<NA>,58.711364,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,621e2e8e67b776a24055b564,2021-05-24,2.0,-1.359378,<NA>,65.40,NaN,<NA>,55.133495,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,621e2e8e67b776a24055b564,2021-05-24,3.0,-0.316295,<NA>,69.01,11.0,<NA>,60.488613,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,621e2e8e67b776a24055b564,2021-05-24,4.0,0.416455,<NA>,65.73,0.0,<NA>,59.110638,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Quickly fit some models without investing too much work:
Include very basic cleaning and imputation in your pipelines (so that the models can actually run), but no feature selection or engineering.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

# Target: predict whether SAD, TENSE/ANXIOUS, or TIRED was reported
TARGETS = ["SAD", "TENSE/ANXIOUS", "TIRED"]

# Exclude identifiers, dates, other mood labels, and location labels
# to prevent data leakage and focus on Fitbit measurements.
exclude = {
    "Unnamed: 0", "id", "date",
    "ALERT", "HAPPY", "NEUTRAL", "RESTED/RELAXED",
    "SAD", "TENSE/ANXIOUS", "TIRED",
    "ENTERTAINMENT", "GYM", "HOME", "HOME_OFFICE",
    "OTHER", "OUTDOORS", "TRANSIT", "WORK/SCHOOL"
}

features = [
    c for c in daily.select_dtypes(include="number").columns
    if c not in exclude
]

# Keep rows where all three target labels are available
data = daily.loc[daily[TARGETS].notna().all(axis=1)].copy()

sad = pd.to_numeric(data["SAD"], errors="coerce")
tense = pd.to_numeric(data["TENSE/ANXIOUS"], errors="coerce")
tired = pd.to_numeric(data["TIRED"], errors="coerce")

valid = sad.notna() & tense.notna() & tired.notna()
data = data.loc[valid]

# Positive class (1): at least one of SAD, TENSE/ANXIOUS, or TIRED
# Negative class (0): none of the three was reported.
y = (
    (sad.loc[valid] > 0)
    | (tense.loc[valid] > 0)
    | (tired.loc[valid] > 0)
).astype(int)

X = data[features]

print("Labeled rows:", len(y))
print("Class counts (0 = none, 1 = at least one mood reported):")
print(y.value_counts())

# Hold out 20% of the data for testing; preserve class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

models = {
    # Dummy baseline: always predicts the majority class. Simple.
    "Dummy baseline": make_pipeline(
        SimpleImputer(strategy="median"),
        DummyClassifier(strategy="most_frequent")
    ),

    # Logistic regression: learns relationships between Fitbit
    # measurements and the target using the training data
    "Logistic regression": make_pipeline(
        SimpleImputer(strategy="median"), ##?? Todo: look into these more
        StandardScaler(), ##?? Todo: look into these more
        LogisticRegression(max_iter=1000, class_weight="balanced")
    )
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    results.append({
        "Model": name,
        # Accuracy: What proportion of all predictions were correct?
        "Accuracy": round(accuracy_score(y_test, pred), 3),
        # Balanced Acc: How well does the model identify both classes, accounting for class imbalance?
        "Balanced accuracy": round(
            balanced_accuracy_score(y_test, pred), 3
        ),
        # Macro F1: How well does the model balance precision and recall across both classes?
        "Macro F1": round(
            f1_score(y_test, pred, average="macro"), 3
        )
    })

display(pd.DataFrame(results))


Labeled rows: 2290
Class counts (0 = none, 1 = at least one mood reported):
1    1292
0     998
Name: count, dtype: int64


,Model,Accuracy,Balanced accuracy,Macro F1
0,Dummy baseline,0.563,0.500,0.360
1,Logistic regression,0.528,0.529,0.526
